# Notebook 02 — Logistic Regression (Baseline)
**Project:** Product Review Sentiment Classification  
**Student:** Eeman Khalid (22F-3173) | BAI-8A  

This notebook trains and evaluates the **Logistic Regression** baseline model using TF-IDF features on the preprocessed Amazon reviews dataset.

> This notebook can run **locally** (no GPU needed) or on Kaggle.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

print('Libraries loaded.')

In [ ]:
# ── Paths — update if running on Kaggle ───────────────────────────────────
# Local:
DATA_PATH      = '../data/processed/amazon_clean.csv'
MODEL_SAVE_DIR = '../models/saved/'

# Kaggle (uncomment if running on Kaggle):
# DATA_PATH      = '/kaggle/input/YOUR-DATASET/amazon_clean.csv'
# MODEL_SAVE_DIR = '/kaggle/working/models/saved/'

os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
print('Paths configured.')

In [ ]:
# ── Load preprocessed data ────────────────────────────────────────────────
df = pd.read_csv(DATA_PATH)
print(f'Loaded {len(df):,} rows')
print(f'Columns: {list(df.columns)}')
print(f'\nClass distribution:')
print(df['sentiment'].value_counts())
df.head(3)

In [ ]:
# ── Train / Val / Test split (70/15/15 stratified) ───────────────────────
X = df['clean_text'].values
y = df['label'].values

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.1765, stratify=y_temp, random_state=42
)

print(f'Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}')

In [ ]:
# ── Build TF-IDF + Logistic Regression Pipeline ───────────────────────────
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=50_000,
        ngram_range=(1, 2),   # unigrams + bigrams
        sublinear_tf=True,    # log normalization
        min_df=3,
        strip_accents='unicode',
        analyzer='word',
    )),
    ('clf', LogisticRegression(
        max_iter=1000,
        C=1.0,
        multi_class='multinomial',
        solver='lbfgs',
        n_jobs=-1,
        random_state=42,
    )),
])

print('Pipeline built.')
print(pipeline)

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────
print('Training Logistic Regression...')
pipeline.fit(X_train, y_train)
print('Training complete.')

In [ ]:
# ── Validation Evaluation ─────────────────────────────────────────────────
LABEL_NAMES = ['negative', 'neutral', 'positive']

y_val_pred = pipeline.predict(X_val)

print('=== VALIDATION RESULTS ===')
print(f"Accuracy  : {accuracy_score(y_val, y_val_pred):.4f}")
print(f"Precision : {precision_score(y_val, y_val_pred, average='weighted', zero_division=0):.4f}")
print(f"Recall    : {recall_score(y_val, y_val_pred, average='weighted', zero_division=0):.4f}")
print(f"F1-Score  : {f1_score(y_val, y_val_pred, average='weighted', zero_division=0):.4f}")
print()
print(classification_report(y_val, y_val_pred, target_names=LABEL_NAMES, zero_division=0))

In [ ]:
# ── Confusion Matrix — Validation ─────────────────────────────────────────
cm = confusion_matrix(y_val, y_val_pred)
plt.figure(figsize=(7,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES)
plt.title('Logistic Regression — Confusion Matrix (Validation)', fontsize=13, fontweight='bold')
plt.ylabel('Actual'); plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig(MODEL_SAVE_DIR + 'lr_confusion_val.png', dpi=150)
plt.show()

In [ ]:
# ── Test Evaluation ───────────────────────────────────────────────────────
y_test_pred = pipeline.predict(X_test)

lr_acc  = accuracy_score(y_test, y_test_pred)
lr_prec = precision_score(y_test, y_test_pred, average='weighted', zero_division=0)
lr_rec  = recall_score(y_test, y_test_pred, average='weighted', zero_division=0)
lr_f1   = f1_score(y_test, y_test_pred, average='weighted', zero_division=0)

print('=== TEST RESULTS ===')
print(f"Accuracy  : {lr_acc:.4f}")
print(f"Precision : {lr_prec:.4f}")
print(f"Recall    : {lr_rec:.4f}")
print(f"F1-Score  : {lr_f1:.4f}")
print()
print(classification_report(y_test, y_test_pred, target_names=LABEL_NAMES, zero_division=0))

In [ ]:
# ── Confusion Matrix — Test ────────────────────────────────────────────────
cm_test = confusion_matrix(y_test, y_test_pred)
plt.figure(figsize=(7,5))
sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues',
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES)
plt.title('Logistic Regression — Confusion Matrix (Test)', fontsize=13, fontweight='bold')
plt.ylabel('Actual'); plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig(MODEL_SAVE_DIR + 'lr_confusion_test.png', dpi=150)
plt.show()

In [ ]:
# ── Top TF-IDF features per class ─────────────────────────────────────────
feature_names = pipeline.named_steps['tfidf'].get_feature_names_out()
coefs = pipeline.named_steps['clf'].coef_

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, (label, ax) in enumerate(zip(LABEL_NAMES, axes)):
    top_idx = coefs[i].argsort()[-20:][::-1]
    top_features = [feature_names[j] for j in top_idx]
    top_weights  = coefs[i][top_idx]
    ax.barh(top_features[::-1], top_weights[::-1], color=['#dc3545','#ffc107','#28a745'][i])
    ax.set_title(f'Top features — {label}', fontweight='bold')
    ax.set_xlabel('TF-IDF Coefficient Weight')
plt.suptitle('Logistic Regression: Most Influential TF-IDF Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(MODEL_SAVE_DIR + 'lr_top_features.png', dpi=150)
plt.show()

In [ ]:
# ── Save model ────────────────────────────────────────────────────────────
model_path = MODEL_SAVE_DIR + 'lr_pipeline.pkl'
joblib.dump(pipeline, model_path)
print(f'Model saved → {model_path}')

import json
lr_results = {
    'Logistic Regression': {
        'accuracy':  round(lr_acc, 4),
        'precision': round(lr_prec, 4),
        'recall':    round(lr_rec, 4),
        'f1':        round(lr_f1, 4),
    }
}

# Save to its OWN file so it never gets overwritten by BiLSTM/BERT results
lr_save_path = MODEL_SAVE_DIR + 'lr_results.json'
with open(lr_save_path, 'w') as f:
    json.dump(lr_results, f, indent=2)
print(f'LR results saved → {lr_save_path}')
print(f'Results: {lr_results}')

In [ ]:
# ── Quick inference test ──────────────────────────────────────────────────
LABEL_MAP_INV = {0: 'negative', 1: 'neutral', 2: 'positive'}

test_reviews = [
    "Absolutely love this product! Works perfectly and arrived fast.",
    "It's okay. Does what it says but nothing special.",
    "Terrible quality. Broke after two days. Complete waste of money."
]

print('=== Inference Test ===')
for review in test_reviews:
    proba = pipeline.predict_proba([review])[0]
    pred  = int(pipeline.predict([review])[0])
    print(f'Review  : {review[:60]}...')
    print(f'Sentiment: {LABEL_MAP_INV[pred].upper()} (confidence: {max(proba):.2%})')
    print(f'Probs   : neg={proba[0]:.3f} | neu={proba[1]:.3f} | pos={proba[2]:.3f}')
    print()